# ✏️ Segmentação Baseada em Bordas — Detector de Canny

**Disciplina:** IA Aplicada à Construção Civil  
**Módulo:** Visão Computacional — Segmentação por Bordas

---

## Contexto

Nos módulos anteriores segmentamos por **intensidade global** (Otsu) e por **similaridade de região** (SLIC+RAG). Agora exploramos uma terceira estratégia: segmentar por **descontinuidades locais de intensidade** — as bordas.

Uma borda ocorre onde o brilho da imagem muda abruptamente. Em pavimentos, isso corresponde a:

- Margem entre asfalto original e remendo (mudança de textura)
- Trincas (linha escura fina num fundo mais claro)
- Juntas de pavimentação
- Limites de faixa de sinalização

### Por que Canny?

O detector de **Canny (1986)** é considerado o padrão-ouro da detecção de bordas por satisfazer três critérios formais simultaneamente:

| Critério | Significado prático |
|----------|--------------------|
| **Boa detecção** | Não perder bordas reais (baixo falso negativo) |
| **Boa localização** | Borda detectada próxima à borda real (erro < 1 px) |
| **Resposta única** | Uma borda real → um único pixel de resposta (sem duplicatas) |

Algoritmos mais simples como Sobel e Laplaciano satisfazem apenas o primeiro critério. O Canny satisfaz os três.

---

## O algoritmo Canny passo a passo

```
Imagem em escala de cinza
        │
        ▼
 1. GaussianBlur (sigma)       ← remove ruído que seria detectado como falsa borda
        │
        ▼
 2. Gradiente de Sobel (Gx, Gy)
    magnitude  = √(Gx² + Gy²)  ← quão forte é a mudança de brilho?
    direção    = arctan(Gy/Gx)  ← em qual direção?
        │
        ▼
 3. Non-Maximum Suppression    ← afina bordas: mantém apenas o pixel de
                                   maior gradiente na direção perpendicular
        │
        ▼
 4. Double Threshold (limiar duplo)
    ├── pixel > threshold2     → borda FORTE  (aceita)
    ├── threshold1 < pixel ≤ threshold2 → borda FRACA (candidata)
    └── pixel ≤ threshold1     → ruído        (rejeita)
        │
        ▼
 5. Hysteresis (rastreamento de bordas)
    Bordas fracas conectadas a bordas fortes → ACEITAS
    Bordas fracas isoladas                  → REJEITADAS
        │
        ▼
  Mapa de bordas binário (0 ou 255)
```

### Os dois limiares — como calibrar

`cv2.Canny(img, threshold1, threshold2)`

| Parâmetro | Papel | Efeito de aumentar |
|-----------|-------|-------------------|
| `threshold1` (baixo) | Limiar para bordas fracas | Menos candidatos → bordas mais curtas |
| `threshold2` (alto) | Limiar para bordas fortes | Apenas mudanças muito abruptas → menos bordas |

**Regra prática:** `threshold2 ≈ 2× threshold1` é um ponto de partida robusto. Para imagens de pavimento (baixo contraste geral), valores em torno de **50/100** funcionam bem. Para imagens com muito ruído, aumente o blur e use limiares mais altos (100/200).

### Comparação com outros detectores

| Detector | Saída | Vantagem | Limitação |
|----------|-------|----------|-----------|
| **Sobel** | Mapa de gradiente (contínuo) | Rápido, simples | Bordas grossas, sensível a ruído |
| **Laplaciano** | Segunda derivada | Detecta zero-crossing | Extremamente sensível a ruído |
| **Canny** | Mapa binário fino | Bordas de 1 px de largura, critérios otimais | Dois limiares a calibrar |
| **Holistically-nested (HED)** | Mapa de probabilidade | Semântico, robusto | Requer modelo treinado |

---

## Estrutura deste notebook

| Seção | O que você vai fazer |
|-------|----------------------|
| 1 | Testar o ambiente com imagem sintética (sem Drive) |
| 2 | Montar o Google Drive e configurar parâmetros |
| 3 | Processar todas as imagens em lote |
| 4 | Visualizar bordas, gradientes e comparativos |
| 5 | Calibrar limiares com grade visual interativa |

> 💡 **Recomendação:** execute a Seção 1 primeiro — ela mostra o efeito do blur e dos limiares sem precisar de nenhuma imagem real.

---
## Seção 1 — Teste com imagem sintética

Criamos uma imagem que simula uma foto de pista com:

- **Dois materiais com brilho diferente** → transição abrupta = borda real
- **Trinca fina** → gradiente alto em largura de 2 px
- **Ruído gaussiano** → falsa borda se blur for insuficiente
- **Faixa branca** → borda de sinalização

**O que esperar:** o Canny deve detectar as bordas do remendo e da trinca com linhas finas de 1 px, **sem** ruído espalhado pelo fundo.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm.notebook import tqdm
import pandas as pd

# ── Criar imagem sintética de pista (600×800, grayscale)
h, w = 480, 640
img_sint = np.full((h, w), 60, dtype=np.uint8)       # asfalto escuro

# Ruído de textura
ruido = np.random.normal(0, 10, (h, w)).astype(np.int16)
img_sint = np.clip(img_sint.astype(np.int16) + ruido, 0, 255).astype(np.uint8)

# Remendo central (asfalto mais novo, mais claro)
img_sint[120:340, 160:480] = np.clip(
    np.full((220, 320), 110, dtype=np.int16)
    + np.random.normal(0, 6, (220, 320)).astype(np.int16),
    0, 255).astype(np.uint8)

# Trinca diagonal sobre o remendo
for i in range(140, 320):
    j = int(180 + (i - 140) * 0.9)
    if j < w:
        img_sint[i, j:j+2] = 20

# Faixa de sinalização branca
img_sint[420:440, 50:590] = 200

# ── Pipeline Canny (parâmetros padrão)
BLUR_K = 5
T1, T2 = 50, 100

img_blur  = cv2.GaussianBlur(img_sint, (BLUR_K, BLUR_K), 1)
edges     = cv2.Canny(img_blur, T1, T2)

# ── Também calcular Sobel para comparação
sobelx = cv2.Sobel(img_blur, cv2.CV_64F, 1, 0, ksize=3)
sobely = cv2.Sobel(img_blur, cv2.CV_64F, 0, 1, ksize=3)
sobel_mag = np.clip(np.sqrt(sobelx**2 + sobely**2), 0, 255).astype(np.uint8)

# ── Visualização
fig, eixos = plt.subplots(1, 4, figsize=(22, 5))
dados = [
    (img_sint,   'Original\n(pista + remendo + trinca + faixa)', 'gray'),
    (img_blur,   f'Após GaussianBlur\n(ksize={BLUR_K}, sigma=1)', 'gray'),
    (sobel_mag,  'Gradiente Sobel\n(magnitude |∇I|)', 'hot'),
    (edges,      f'Bordas Canny\n(T1={T1}, T2={T2})', 'gray'),
]
for ax, (im, t, cmap) in zip(eixos, dados):
    ax.imshow(im, cmap=cmap)
    ax.set_title(t, fontsize=9)
    ax.axis('off')

n_borda = int(edges.sum() // 255)
pct_borda = 100 * n_borda / edges.size
plt.suptitle(
    f'✅ Teste sintético — Canny  |  {n_borda} pixels de borda  ({pct_borda:.2f}% da imagem)',
    fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()
print(f'Pixels de borda detectados : {n_borda}  ({pct_borda:.2f}%)')
print('✅ Ambiente OK — pode prosseguir para a Seção 2.')

---
## Seção 2 — Montar o Google Drive e configurar

> ⚠️ **Pasta compartilhada?** Se `fotos_obra` não estiver em *Meu Drive*:  
> *Drive → clique direito → Organizar → Adicionar atalho ao Drive*

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive montado.')

In [ ]:
# ── Caminhos
PASTA_ENTRADA = Path('/content/drive/MyDrive/fotos_obra')
PASTA_SAIDA   = Path('/content/drive/MyDrive/fotos_obra_canny')
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)

# ── Parâmetros Canny
BLUR_KSIZE  = 5      # kernel GaussianBlur (ímpar, 3–11)
BLUR_SIGMA  = 1      # desvio padrão do blur (0 = calculado automaticamente)
THRESHOLD1  = 50     # limiar baixo (bordas fracas)
THRESHOLD2  = 100    # limiar alto  (bordas fortes) — recomendado ≈ 2×T1

# ── Redimensionamento
LARGURA_MAX = 1200

# ── Extensões
EXTENSOES = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}

if not PASTA_ENTRADA.exists():
    print(f'⛔ Pasta não encontrada: {PASTA_ENTRADA}')
else:
    arquivos = sorted([p for p in PASTA_ENTRADA.iterdir() if p.suffix.lower() in EXTENSOES])
    print(f'📂 Entrada   : {PASTA_ENTRADA}')
    print(f'📁 Saída     : {PASTA_SAIDA}')
    print(f'⚙️  Blur      : ksize={BLUR_KSIZE}, sigma={BLUR_SIGMA}')
    print(f'⚙️  Canny     : T1={THRESHOLD1}, T2={THRESHOLD2}  (razão T2/T1 = {THRESHOLD2/THRESHOLD1:.1f})')
    print(f'🖼️  {len(arquivos)} imagem(ns) encontrada(s)')

---
## Seção 3 — Processamento em lote

Para cada imagem são salvos 4 arquivos:

```
fotos_obra_canny/
  nome_foto_blur.png         ← imagem após GaussianBlur
  nome_foto_sobel.png        ← mapa de magnitude do gradiente Sobel
  nome_foto_canny.png        ← mapa de bordas Canny (binário)
  nome_foto_overlay.png      ← bordas sobrepostas à imagem original (vermelho)
  nome_foto_comparativo.png  ← painel 4 vistas
  metricas_canny.csv         ← % de pixels de borda por imagem
```

> 💡 **O overlay** (bordas em vermelho sobre o original) é o arquivo mais útil para relatório técnico — permite visualizar onde o Canny detectou transições sem perder o contexto da imagem.

In [ ]:
def processar_canny(img_gray, blur_ksize=5, blur_sigma=1, t1=50, t2=100):
    """Aplica blur + Canny + Sobel. Retorna (blur, sobel_mag, edges)."""
    ksize = max(3, blur_ksize | 1)   # garante ímpar
    blur  = cv2.GaussianBlur(img_gray, (ksize, ksize), blur_sigma)
    edges = cv2.Canny(blur, t1, t2)

    sobelx    = cv2.Sobel(blur, cv2.CV_64F, 1, 0, ksize=3)
    sobely    = cv2.Sobel(blur, cv2.CV_64F, 0, 1, ksize=3)
    sobel_mag = np.clip(np.sqrt(sobelx**2 + sobely**2), 0, 255).astype(np.uint8)

    return blur, sobel_mag, edges


todas_stats = []
erros       = []

for arq in tqdm(arquivos, desc='Detectando bordas'):
    try:
        img_bgr = cv2.imread(str(arq))
        assert img_bgr is not None

        if LARGURA_MAX > 0 and img_bgr.shape[1] > LARGURA_MAX:
            r       = LARGURA_MAX / img_bgr.shape[1]
            img_bgr = cv2.resize(img_bgr, (LARGURA_MAX, int(img_bgr.shape[0] * r)),
                                 interpolation=cv2.INTER_AREA)

        gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
        blur, sobel_mag, edges = processar_canny(
            gray, BLUR_KSIZE, BLUR_SIGMA, THRESHOLD1, THRESHOLD2)

        # Overlay: bordas em vermelho sobre imagem original
        overlay = img_bgr.copy()
        overlay[edges > 0] = [0, 0, 220]   # vermelho BGR

        # Métricas
        n_borda   = int(edges.sum() // 255)
        pct_borda = round(100 * n_borda / edges.size, 3)
        todas_stats.append({
            'arquivo'       : arq.name,
            'largura'       : img_bgr.shape[1],
            'altura'        : img_bgr.shape[0],
            'T1'            : THRESHOLD1,
            'T2'            : THRESHOLD2,
            'blur_ksize'    : BLUR_KSIZE,
            'pixels_borda'  : n_borda,
            'pct_borda'     : pct_borda,
        })

        # Painel comparativo
        def gray3(m): return cv2.cvtColor(m, cv2.COLOR_GRAY2BGR)
        sobel_bgr = cv2.applyColorMap(sobel_mag, cv2.COLORMAP_HOT)
        painel_top = np.hstack([img_bgr, gray3(blur)])
        painel_bot = np.hstack([sobel_bgr, overlay])
        painel     = np.vstack([painel_top, painel_bot])

        cv2.rectangle(painel, (10, 10), (720, 62), (255, 255, 255), -1)
        cv2.putText(painel,
                    f'blur={BLUR_KSIZE}  T1={THRESHOLD1}  T2={THRESHOLD2}  '
                    f'bordas={n_borda} px ({pct_borda:.2f}%)',
                    (18, 48), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (20, 20, 20), 2, cv2.LINE_AA)

        base = arq.stem
        cv2.imwrite(str(PASTA_SAIDA / f'{base}_blur.png'),        gray3(blur))
        cv2.imwrite(str(PASTA_SAIDA / f'{base}_sobel.png'),       sobel_bgr)
        cv2.imwrite(str(PASTA_SAIDA / f'{base}_canny.png'),       edges)
        cv2.imwrite(str(PASTA_SAIDA / f'{base}_overlay.png'),     overlay)
        cv2.imwrite(str(PASTA_SAIDA / f'{base}_comparativo.png'), painel)

    except Exception as e:
        erros.append((arq.name, str(e)))

df = pd.DataFrame(todas_stats)
df.to_csv(str(PASTA_SAIDA / 'metricas_canny.csv'), index=False)

print(f'\n✅ {len(todas_stats)} imagem(ns) processada(s)')
if len(df) > 0:
    print(f'   % borda média  : {df["pct_borda"].mean():.3f}%')
    print(f'   % borda máxima : {df["pct_borda"].max():.3f}%  ({df.loc[df["pct_borda"].idxmax(), "arquivo"]})')
print(f'   Relatório      : {PASTA_SAIDA / "metricas_canny.csv"}')
if erros:
    print('\n⚠️  Erros:')
    for n, m in erros: print(f'   {n}: {m}')

---
## Seção 4 — Visualização: bordas, gradientes e overlay

### 4a. Painel de quatro vistas

| Quadrante | O que analisar |
|-----------|----------------|
| **Original** | Referência |
| **Após blur** | O ruído de alta frequência deve ter sumido. Se detalhes importantes foram borrados, reduza `BLUR_KSIZE` |
| **Gradiente Sobel** | Mapa de calor — regiões quentes = mudanças abruptas. Trincas e juntas devem aparecer em laranja/branco |
| **Overlay Canny** | Bordas em vermelho sobre a imagem. Verificar: cobre as trincas? Tem muito ruído no fundo? |

In [ ]:
MAX_EXIBIR = 4

for arq in arquivos[:MAX_EXIBIR]:
    img_bgr = cv2.imread(str(arq))
    if img_bgr is None: continue
    if LARGURA_MAX > 0 and img_bgr.shape[1] > LARGURA_MAX:
        r = LARGURA_MAX / img_bgr.shape[1]
        img_bgr = cv2.resize(img_bgr, (LARGURA_MAX, int(img_bgr.shape[0]*r)),
                             interpolation=cv2.INTER_AREA)

    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    blur, sobel_mag, edges = processar_canny(gray, BLUR_KSIZE, BLUR_SIGMA, THRESHOLD1, THRESHOLD2)

    overlay = img_bgr.copy()
    overlay[edges > 0] = [0, 0, 220]

    n_borda   = int(edges.sum() // 255)
    pct_borda = 100 * n_borda / edges.size

    fig, eixos = plt.subplots(1, 4, figsize=(22, 5))
    dados = [
        (cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB), 'Original', None),
        (blur,      f'GaussianBlur\nksize={BLUR_KSIZE}, sigma={BLUR_SIGMA}', 'gray'),
        (sobel_mag, 'Gradiente Sobel\n(magnitude |∇I|)', 'hot'),
        (cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB),
                    f'Overlay Canny (T1={THRESHOLD1}, T2={THRESHOLD2})\n'
                    f'{n_borda} px de borda ({pct_borda:.2f}%)', None),
    ]
    for ax, (im, t, cmap) in zip(eixos, dados):
        ax.imshow(im, cmap=cmap)
        ax.set_title(t, fontsize=9)
        ax.axis('off')

    plt.suptitle(arq.name, fontsize=10, fontweight='bold')
    plt.tight_layout()
    plt.show()

### 4b. Perfil de gradiente — corte transversal na trinca

Este gráfico extrai uma **linha horizontal** da imagem e plota o perfil de intensidade e gradiente ao longo dela. O pico do gradiente deve coincidir exatamente com a borda visual da trinca.

É a forma mais direta de entender o que o Canny está enxergando — e de diagnosticar por que uma borda não foi detectada (gradiente abaixo de T2) ou por que há falsos positivos (ruído com gradiente alto).

In [ ]:
if arquivos:
    img_bgr = cv2.imread(str(arquivos[0]))
    if LARGURA_MAX > 0 and img_bgr.shape[1] > LARGURA_MAX:
        r = LARGURA_MAX / img_bgr.shape[1]
        img_bgr = cv2.resize(img_bgr, (LARGURA_MAX, int(img_bgr.shape[0]*r)),
                             interpolation=cv2.INTER_AREA)

    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    blur, sobel_mag, edges = processar_canny(gray, BLUR_KSIZE, BLUR_SIGMA, THRESHOLD1, THRESHOLD2)

    # Linha de corte central
    linha = gray.shape[0] // 2
    perfil_gray   = gray[linha, :].astype(np.float32)
    perfil_grad   = sobel_mag[linha, :].astype(np.float32)
    perfil_edges  = edges[linha, :]

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

    ax1.plot(perfil_gray, color='steelblue', lw=1.5, label='Intensidade')
    ax1.set_ylabel('Intensidade (0–255)')
    ax1.set_title(f'Perfil horizontal — linha y={linha}  |  {arquivos[0].name}', fontsize=10)
    ax1.legend(loc='upper right')
    ax1.grid(alpha=0.3)

    ax2.plot(perfil_grad, color='darkorange', lw=1.5, label='|∇I| Sobel')
    ax2.axhline(THRESHOLD1, color='blue',   lw=1, ls='--', label=f'T1={THRESHOLD1} (borda fraca)')
    ax2.axhline(THRESHOLD2, color='red',    lw=1, ls='--', label=f'T2={THRESHOLD2} (borda forte)')
    # Marcar posições Canny
    pos_borda = np.where(perfil_edges > 0)[0]
    ax2.scatter(pos_borda, perfil_grad[pos_borda], color='red', zorder=5,
                s=40, label=f'Canny detectou ({len(pos_borda)} pontos)')
    ax2.set_ylabel('Magnitude do gradiente')
    ax2.set_xlabel('Posição x (pixels)')
    ax2.legend(loc='upper right')
    ax2.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()
    print(f'Linha analisada : y = {linha}')
    print(f'Bordas Canny na linha : {len(pos_borda)} ponto(s)  em x = {pos_borda.tolist()[:10]}')

### 4c. Relatório — ranking por densidade de bordas

A **densidade de bordas** (% de pixels detectados) é uma métrica de complexidade visual da imagem. Valores altos podem indicar:
- Muitas patologias na superfície ✅
- Excesso de ruído → limiares muito baixos ⚠️
- Textura muito heterogênea (brita exposta, agregado solto) ⚠️

In [ ]:
df_exib = df[['arquivo', 'largura', 'altura', 'pixels_borda', 'pct_borda']]\
            .sort_values('pct_borda', ascending=False).reset_index(drop=True)
df_exib.index += 1

def colorir_pct(val):
    if   val > 15 : return 'background-color: #ffcccc'   # vermelho — possível ruído
    elif val > 5  : return 'background-color: #fff3cd'   # amarelo — verificar
    else          : return 'background-color: #d4edda'   # verde — normal

display(df_exib.style.applymap(colorir_pct, subset=['pct_borda']))

print('\n🟢 Verde   : < 5%  — densidade normal para pavimento')
print('🟡 Amarelo : 5–15% — verificar se é patologia real ou ruído')
print('🔴 Vermelho: > 15% — provável excesso de ruído → aumentar blur ou limiares')

---
## Seção 5 — Calibração: grade de blur × limiares

### Estratégia de calibração

Calibre nesta ordem:

1. **`BLUR_KSIZE`** primeiro — um blur inadequado não pode ser corrigido pelos limiares
2. **`THRESHOLD2`** — controla quantas bordas fortes são aceitas (mais crítico)
3. **`THRESHOLD1`** — mantendo a razão T2/T1 ≈ 2, ajusta a extensão das bordas fracas

A grade abaixo varia `BLUR_KSIZE` (linhas) × `THRESHOLD2` (colunas), mantendo T1 = T2/2.

In [ ]:
if arquivos:
    img_bgr = cv2.imread(str(arquivos[0]))
    if LARGURA_MAX > 0 and img_bgr.shape[1] > LARGURA_MAX:
        r = LARGURA_MAX / img_bgr.shape[1]
        img_bgr = cv2.resize(img_bgr, (LARGURA_MAX, int(img_bgr.shape[0]*r)),
                             interpolation=cv2.INTER_AREA)
    gray_ref = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

    blur_vals = [3, 5, 9]
    t2_vals   = [50, 100, 150, 200]

    fig, eixos = plt.subplots(
        len(blur_vals), len(t2_vals) + 1,
        figsize=(5 * (len(t2_vals)+1), 4.5 * len(blur_vals))
    )

    for i, bk in enumerate(blur_vals):
        # Coluna 0 = original
        eixos[i][0].imshow(gray_ref, cmap='gray')
        eixos[i][0].set_ylabel(f'blur ksize={bk}', fontsize=10, fontweight='bold')
        eixos[i][0].set_title('Original' if i == 0 else '', fontsize=9)
        eixos[i][0].axis('off')

        for j, t2 in enumerate(t2_vals):
            t1   = t2 // 2
            _, _, ed = processar_canny(gray_ref, bk, BLUR_SIGMA, t1, t2)
            pct  = round(100 * int(ed.sum()//255) / ed.size, 2)

            # Overlay
            ov = cv2.cvtColor(gray_ref, cv2.COLOR_GRAY2BGR)
            ov[ed > 0] = [0, 0, 220]

            eixos[i][j+1].imshow(cv2.cvtColor(ov, cv2.COLOR_BGR2RGB))
            eixos[i][j+1].set_title(
                f'T1={t1}, T2={t2}\n{pct:.1f}% borda' if i == 0
                else f'{pct:.1f}% borda', fontsize=9)
            eixos[i][j+1].axis('off')

    plt.suptitle(f'Grade de calibração Canny — {arquivos[0].name}',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()
    print('👆 Escolha a combinação com bordas nítidas das trincas/juntas e mínimo ruído no fundo.')
    print('   Atualize BLUR_KSIZE, THRESHOLD1 e THRESHOLD2 na Seção 2 e reprocesse.')

---
## Próximos passos

### Posição do Canny no pipeline completo

```
Imagem bruta
    │
    ├─ [módulo 03] CLAHE           → contraste local melhorado
    ├─ [módulo 07] Remoção artefatos → fundo limpo
    │
    ▼
  Canny (este módulo)
    │  mapa de bordas
    ├─── Dilate + findContours → mede comprimento de trinca [px → m]
    ├─── Hough Lines (cv2.HoughLinesP) → detecta trincas longitudinais/transversais
    └─── Entrada para U-Net como canal adicional (RGB + bordas = 4 canais)
```

### Quando usar Canny — e quando não usar

| Situação | Canny | Alternativa |
|----------|-------|-------------|
| Trinca fina em fundo uniforme | ✅ Excelente | — |
| Borda de remendo asfáltico | ✅ Bom | SLIC+RAG (mais robusto a textura) |
| Superfície com brita exposta | ⚠️ Muito ruído | Aumentar blur + limiares altos |
| Imagem úmida (reflexo de água) | ❌ Falsos positivos | Filtro bilateral antes do Canny |
| Detecção semântica (tipo de patologia) | ❌ Não classifica | U-Net, YOLOv8 |

### Referências

- Canny, J. (1986). *A Computational Approach to Edge Detection*. IEEE TPAMI, 8(6), 679–698.
- Marr, D. & Hildreth, E. (1980). *Theory of Edge Detection*. Proc. Royal Society B, 207(1167). — base teórica do Laplaciano
- OpenCV docs: [`cv2.Canny`](https://docs.opencv.org/4.x/da/d22/tutorial_py_canny.html), [`cv2.Sobel`](https://docs.opencv.org/4.x/d2/d2c/tutorial_sobel_derivatives.html)
- DNIT (2006). *Manual de Restauração de Pavimentos Asfálticos* — classificação de trincas longitudinais e transversais
